In [ ]:
import json
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def load_inverted_index(filename):
    with open(filename, 'r') as file:
        data = json.load(file)
    inverted_index = {}
    for entry in data:
        doc_ids = []
        for doc_str in entry["ids"].split(","):
            doc_str = doc_str.strip()
            if doc_str.isdigit():
                doc_ids.append(int(doc_str))
        inverted_index[entry["word"]] = doc_ids
    return inverted_index

def load_corpus(filename):
    df = pd.read_excel(filename, engine='openpyxl')
    corpus = {}
    for _, row in df.iterrows():
        doc_id = str(row["Document ID"])
        tokens = str(row["Tokens"])
        corpus[doc_id] = tokens
    return corpus

def load_queries(filename):
    queries = []
    with open(filename, 'r') as file:
        for line in file:
            entry = json.loads(line.strip())
            q_id = str(entry["_id"])
            q_text = entry["text"].lower()
            queries.append((q_id, q_text))
    return queries

def compute_tfidf_top100(query, inverted_index, corpus):

    query_tokens = query.split()
    relevant_docs = set()
    for token in query_tokens:
        if token in inverted_index:
            for docid in inverted_index[token]:
                relevant_docs.add(str(docid))

    if not relevant_docs:
        return []

    doc_texts = [corpus[doc_id] for doc_id in relevant_docs if doc_id in corpus]
    doc_ids = [doc_id for doc_id in relevant_docs if doc_id in corpus]

    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform([query] + doc_texts)

    query_vector = vectors[0:1]
    doc_vectors = vectors[1:]

    sims = cosine_similarity(query_vector, doc_vectors)[0]

    scored_docs = list(zip(doc_ids, sims))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    return scored_docs[:100]

def load_neural_model():

    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    return model

def neural_rerank(query_text, top_docs, corpus, model):

    query_emb = model.encode(query_text, convert_to_numpy=True)

    reranked = []
    for (doc_id, _) in top_docs:
        doc_text = corpus.get(doc_id, "")
        doc_emb = model.encode(doc_text, convert_to_numpy=True)
        dot = np.dot(query_emb, doc_emb)
        norm_q = np.linalg.norm(query_emb)
        norm_d = np.linalg.norm(doc_emb)
        cos_sim = dot / (norm_q * norm_d + 1e-10)
        reranked.append((doc_id, cos_sim))

    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked

def main():
    inverted_index = load_inverted_index(r"scifact\inverted_index.json")
    corpus = load_corpus(r"scifact\preprocessed_corpus.xlsx")
    all_queries = load_queries(r"scifact\queries.jsonl") 
    sbert_model = load_neural_model()
    test_queries = []
    for q_id, q_text in all_queries:
        q_int = int(q_id)
        if q_int % 2 == 1 and q_int <= 1395:
            test_queries.append((q_id, q_text))
    
    print(f"Total queries loaded: {len(all_queries)}. Test queries to process: {len(test_queries)}.")

    output_file = r"scifact\assignment2_results.txt"
    run_name = "neural_run"

    with open(output_file, "w", encoding="utf-8") as outf:
        for query_id, query_text in test_queries:
            tfidf_top100 = compute_tfidf_top100(query_text, inverted_index, corpus)
            final_ranked = neural_rerank(query_text, tfidf_top100, corpus, sbert_model)
            for rank, (doc_id, score) in enumerate(final_ranked, start=1):
                outf.write(f"{query_id} Q0 {doc_id} {rank} {score:.4f} {run_name}\n")

    print(f"Done! Final results (only odd queries up to 1395) saved to {output_file}")

if __name__ == "__main__":
    main()


Total queries loaded: 1109. Test queries to process: 542.
Done! Final results (only odd queries up to 1395) saved to scifact\assignment2_results.txt


In [13]:
import pandas as pd
from sklearn.metrics import average_precision_score

def compute_map(results_file, qrels_file):

    results_df = pd.read_csv(results_file, sep=" ", header=None, names=["query_id", "Q0", "doc_id", "rank", "score", "tag"])
    qrels_df = pd.read_csv(qrels_file, sep=" ", header=None, names=["query_id", "Q0", "doc_id", "relevance"])
    query_ids = results_df["query_id"].unique()
    average_precisions = []

    for query_id in query_ids:
        retrieved_docs = results_df[results_df["query_id"] == query_id]["doc_id"].tolist()
        relevant_docs = qrels_df[qrels_df["query_id"] == query_id]["doc_id"].tolist()
        y_true = [1 if doc in relevant_docs else 0 for doc in retrieved_docs]

        if sum(y_true) > 0:
            ap = average_precision_score(y_true, list(range(len(y_true), 0, -1)))
            average_precisions.append(ap)

    map_score = sum(average_precisions) / len(average_precisions) if average_precisions else 0
    print(f"Mean Average Precision (MAP): {map_score:.4f}")

compute_map(r"scifact\assignment2_results.txt", r"scifact\qrels.txt")

Mean Average Precision (MAP): 0.5916
